In [0]:
%run ./01_logging_utils.py

**schema drift detection and quarantine**

In [0]:

from datetime import datetime
import json

from pyspark.sql import functions as F


def quarantine_schema_drift(path, batch_id=None):
    """
    Inspect a raw JSON file and quarantine records with
    unexpected or missing fields.
    """
    run_id = new_batch_id("schema_drift")
    start_time = datetime.now()

    if batch_id is None:
        batch_id = new_batch_id("drift")

    try:
        # Read the original JSON file
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)

        records = data.get("data", [])

        expected_keys = EXP_KEYS
        quarantined = []

        for record in records:
            if not isinstance(record, dict):
                quarantined.append((
                    json.dumps(record),
                    "record is not a JSON object",
                    path,
                    batch_id,
                    datetime.now()
                ))
                continue

            actual_keys = set(record.keys())

            if actual_keys != expected_keys:
                extra = sorted(actual_keys - expected_keys)
                missing = sorted(expected_keys - actual_keys)

                reason = (
                    f"schema drift: extra={extra}, missing={missing}"
                )

                quarantined.append((
                    json.dumps(record),
                    reason,
                    path,
                    batch_id,
                    datetime.now()
                ))

        if quarantined:
            quarantine_df = spark.createDataFrame(
                quarantined,
                QUAR_SCHEMA_DEF
            )

            quarantine_df.createOrReplaceTempView(
                "schema_drift_source"
            )

            spark.sql(f"""
                MERGE INTO {T_QUAR} AS target
                USING schema_drift_source AS source
                ON target.raw_record = source.raw_record
                   AND target.source_file = source.source_file
                WHEN NOT MATCHED THEN INSERT *
            """)

        log_run(
            run_id,
            "schema_drift",
            f"path={path}, batch_id={batch_id}",
            start_time,
            datetime.now(),
            "SUCCESS",
            len(quarantined),
            0
        )

        print("Schema drift scan completed.")
        print(f"Records inspected: {len(records)}")
        print(f"Records quarantined: {len(quarantined)}")

    except Exception as e:
        log_run(
            run_id,
            "schema_drift",
            f"path={path}, batch_id={batch_id}",
            start_time,
            datetime.now(),
            "FAILED",
            err=str(e)[:500]
        )
        raise


In [0]:
quarantine_schema_drift(FULL_FILE)

In [0]:
display(spark.sql("""
    SELECT reason, COUNT(*) AS record_count
    FROM crypto.bronze.quarantine
    GROUP BY reason
    ORDER BY record_count DESC
"""))

In [0]:
display(spark.sql("""
    SELECT raw_record, reason, source_file, batch_id
    FROM crypto.bronze.quarantine
    ORDER BY load_timestamp DESC
    LIMIT 20
"""))